## Configure OpenAI API Key


In [1]:
import os
from getpass import getpass
from openai import OpenAI 

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key")
    
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])   

print("OpenAi Client configured successfully")
    
    

OpenAi Client configured successfully


## PDF Path

In [3]:

from pathlib import Path

# 1. PDF file path
pdf_path = Path(
    r"D:\JEEVA\GENAI\MVP_RAG\Resume.pdf"
)

# 2. Check whether the PDF exists
if pdf_path.exists():
    print("PDF found:", pdf_path.name)
    print("File size:", pdf_path.stat().st_size, "bytes")
else:
    print("PDF file not found. Please check the file path.")


PDF found: Resume.pdf
File size: 290226 bytes


## PDF Reader

In [5]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
pages=[]

for page_numser, page in enumerate(reader.pages,start=1):
    page_text = page.extract_text() or ""
    pages.append({
        "page":page_numser,
        "text":page_text
    })

text = "\n".join(page["text"] for page in pages)

print("pages:", len(pages))
print("character extracted:",len(text))
print("\nPreview:\n")
print(text[:1000])


pages: 1
character extracted: 3499

Preview:

JEEVA E
+91 8870147970 | jeevae3112@gmail.com | linkedin.com/in/jeeva | github.com/jeeva | Chennai
SUMMARY
Python Full Stack Developer with 2 years of experience building healthcare and industrial applications using Python, Django,
FastAPI, React.js, PostgreSQL, MySQL, Redis, and WebSockets. Experienced in developing REST APIs, real-time systems,
asynchronous workflows, role-based access control, and microservice-based architectures. Hands-on experience in Generative AI
and LLM application development, including conversational AI, tool/function calling, AI-driven API integration, prompt
engineering, and context management.
TECHNICAL SKILLS
Languages:Python, JavaScript
Backend:Django, Django REST Framework, FastAPI, Flask
Frontend:React.js, Redux
AI & Integrations:LLMs, Prompt Engineering, Function Calling, Conversational State Management
Databases & Cache:PostgreSQL, MySQL, Redis
Messaging & Async:Celery, WebSockets, Django Channels
DevOps 

## Chunking

#### simple character-based chunking strategy

In [7]:
chunk_size = 500
overlap = 50

chunks = []
start=0
chunk_id = 0

while start < len(text):
    end= start+ chunk_size
    chunk_text = text[start:end].strip()
    
    if chunk_text:
        chunks.append({
            "id":f"chunk-{chunk_id}",
            "text":chunk_text
        })
        chunk_id+=1
        
    start += chunk_size-overlap
    
print("total chunks:",len(chunks))
print("\n First chunk:\n")
print(chunks[0]["text"])



total chunks: 8

 First chunk:

JEEVA E
+91 8870147970 | jeevae3112@gmail.com | linkedin.com/in/jeeva | github.com/jeeva | Chennai
SUMMARY
Python Full Stack Developer with 2 years of experience building healthcare and industrial applications using Python, Django,
FastAPI, React.js, PostgreSQL, MySQL, Redis, and WebSockets. Experienced in developing REST APIs, real-time systems,
asynchronous workflows, role-based access control, and microservice-based architectures. Hands-on experience in Generative AI
and LLM application devel


## Embedding

In [11]:
# EMBEDDING_MODEL = "text-embedding-3-small"

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(EMBEDDING_MODEL)

def get_embeddings(texts, batch_size=100):
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True
    )
    return embeddings

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_embeddings = get_embeddings(chunk_texts)


print("Number of Embedding:", len(chunk_embeddings))
print("Embedding dimentions:",len(chunk_embeddings[0]))
print("First 10 Values:",chunk_embeddings[0][:10])





d:\JEEVA\GENAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.59it/s]

Number of Embedding: 8
Embedding dimentions: 384
First 10 Values: [-0.14421986  0.03112516 -0.011597   -0.00252082 -0.08554696 -0.11193999
  0.00129448  0.08630208 -0.04681053 -0.06846756]


## Other VectorDBs and why Chroma?

Vector DB = Simple Understanding
ChromaDB = Very Beginner-friendly; great for learning
Pinecone = Managed/cloud-oriented Vector Database
Qdrant = Open-source/Vector Search Focused
Weaviate = Vector database with Broader Search Capabilities
Milvus = Designed for large-scale Vector Workloads
pgvector = Vector Search inside PostgreSQL




### Store in Vector Database

In [16]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="Resume")

existing = collection.get()
if existing["ids"]:
    collection.delete(ids=existing["ids"])
    
    
collection.add(
    ids=[chunk["id"] for chunk in chunks],
    documents=[chunk["text"] for chunk in chunks],
    embeddings = chunk_embeddings,
    metadatas = [
        {"source": str(pdf_path)} 
        for _ in chunks
        ]
    )

print("chunks stored in vector Database:", collection.count())




chunks stored in vector Database: 8


## Chroma Supports
### Storing....

#### Documents, embeddings,metadata,IDs together in a collections


## User Query ===> Query Embedding

In [17]:
question = "Breifly about her projects?"

question_embedding = get_embeddings([question])[0]
print("Question:",question)
print("Query Embedding Dimention",len(question_embedding))



Batches: 100%|██████████| 1/1 [00:00<00:00, 15.58it/s]

Question: Breifly about her projects?
Query Embedding Dimention 384


In [20]:
results = collection.query(query_embeddings=[question_embedding],n_results=3)
retrived_chunks = results["documents"][0]
retrieved_ids = results["distances"][0]
distance = results["distances"][0]
print("Retrieved Chunk: \n")

for i,(chunk_id,chunk_text,distance) in enumerate(zip(retrieved_ids,retrived_chunks,distance),start=1):
    print(f"-----Result {i} | {chunk_id} | distance={distance:.4f}----")
    print(chunk_text[:1000])
    print()




Retrieved Chunk: 

-----Result 1 | 1.5285381078720093 | distance=1.5285----
erience in Generative AI
and LLM application development, including conversational AI, tool/function calling, AI-driven API integration, prompt
engineering, and context management.
TECHNICAL SKILLS
Languages:Python, JavaScript
Backend:Django, Django REST Framework, FastAPI, Flask
Frontend:React.js, Redux
AI & Integrations:LLMs, Prompt Engineering, Function Calling, Conversational State Management
Databases & Cache:PostgreSQL, MySQL, Redis
Messaging & Async:Celery, WebSockets, Django Channels
Dev

-----Result 2 | 1.625630497932434 | distance=1.6256----
r secure login, and used Celery with Redis for asynchronous appointment notifications
and follow-up workflows.
EDUCATION
King Nandhivarman College of Arts and Science
Bachelor of Computer Applications (BCA) 2019 – 2022
CERTIFICATIONS
Besant Technologies
Full Stack Web Development Certificate - React.js, Python, Django, MySQL July 2022 – March 2023

-----Result 3 |

In [21]:
results

{'ids': [['chunk-1', 'chunk-7', 'chunk-0']],
 'embeddings': None,
 'documents': [['erience in Generative AI\nand LLM application development, including conversational AI, tool/function calling, AI-driven API integration, prompt\nengineering, and context management.\nTECHNICAL SKILLS\nLanguages:Python, JavaScript\nBackend:Django, Django REST Framework, FastAPI, Flask\nFrontend:React.js, Redux\nAI & Integrations:LLMs, Prompt Engineering, Function Calling, Conversational State Management\nDatabases & Cache:PostgreSQL, MySQL, Redis\nMessaging & Async:Celery, WebSockets, Django Channels\nDev',
   'r secure login, and used Celery with Redis for asynchronous appointment notifications\nand follow-up workflows.\nEDUCATION\nKing Nandhivarman College of Arts and Science\nBachelor of Computer Applications (BCA) 2019 – 2022\nCERTIFICATIONS\nBesant Technologies\nFull Stack Web Development Certificate - React.js, Python, Django, MySQL July 2022 – March 2023',
   'JEEVA E\n+91 8870147970 | jeevae3112@

## Prompt + Retrieved Context

In [22]:
context = "\n\n ---------- Retrieved Chunk ------\n\n".join(retrived_chunks)

prompt = f"""
Answer the User's question using only the retrieved context below.
If the Answer is not Present in the context, Say that the information is not available in the provided document.
Retrieved Context: {context}
User Question:{question}

""".strip()

print(prompt)



Answer the User's question using only the retrieved context below.
If the Answer is not Present in the context, Say that the information is not available in the provided document.
Retrieved Context: erience in Generative AI
and LLM application development, including conversational AI, tool/function calling, AI-driven API integration, prompt
engineering, and context management.
TECHNICAL SKILLS
Languages:Python, JavaScript
Backend:Django, Django REST Framework, FastAPI, Flask
Frontend:React.js, Redux
AI & Integrations:LLMs, Prompt Engineering, Function Calling, Conversational State Management
Databases & Cache:PostgreSQL, MySQL, Redis
Messaging & Async:Celery, WebSockets, Django Channels
Dev

 ---------- Retrieved Chunk ------

r secure login, and used Celery with Redis for asynchronous appointment notifications
and follow-up workflows.
EDUCATION
King Nandhivarman College of Arts and Science
Bachelor of Computer Applications (BCA) 2019 – 2022
CERTIFICATIONS
Besant Technologies
Full Stac

In [ ]:
GENERATION_MODEL = "gpt-4.1-mini"

respons = client.responses.create(model=GENERATION_MODEL,input=prompt)
answer = respons.output_text

print("Answer: \n\n")
print(answer)


NotFoundError: Error code: 404 - The model `gemini-2.5-flash-lite` does not exist or you do not have access to it.

## 2 Points to Note

### 1. The role of the vector Database?

#### does not generate answer , to store embeddings and retrieve
#### the most relevant chunks for the user's query

### 2. The Role of the LLM
#### llm receives the retrieved context and generates the final answer.

